In [3]:
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings('ignore')

# 경로 설정 (현재 노트북 위치 기준)
file_path = 'disease_weather_air_merged.csv'  # 파일이 같은 폴더에 있다고 가정
if not os.path.exists(file_path):
  file_path = './combined_data/disease_weather_air_merged.csv'

df = pd.read_csv(file_path)

# 날짜 파싱 및 계절성 파생변수 생성
df['진료년월_dt'] = pd.to_datetime(df['진료년월'].astype(str), errors='coerce')
df['연도'] = df['진료년월_dt'].dt.year
df['월'] = df['진료년월_dt'].dt.month

df['sin1'] = np.sin(2 * np.pi * df['월'] / 12)
df['cos1'] = np.cos(2 * np.pi * df['월'] / 12)

# 2015년 이후, 코로나 기간(2020~2022) 제외 필터링
df_filtered = df[(df['연도'] >= 2015) & (~df['연도'].between(2020, 2022))].copy()
print('전처리 및 필터링 완료. 데이터 크기:', df_filtered.shape)

전처리 및 필터링 완료. 데이터 크기: (801, 28)


In [4]:
# L23 데이터만 필터링
l23_df = df_filtered[df_filtered['질병코드'] == 'L23'].copy()
l23_df = l23_df.dropna(
    subset=[
        '환자수',
        '월평균기온',
        '월평균습도',
        '월평균_일교차',
        '월평균_PM25',
        'sin1',
        'cos1',
    ]
)

print(f'L23 유효 데이터 개수: {len(l23_df)}개')

# 1. 습도와 환자 수 간의 단순 상관계수 확인
corr_humid = l23_df['월평균습도'].corr(l23_df['환자수'])
print(f'👉 L23 환자수와 월평균습도의 단순 상관계수: {corr_humid:.4f}')

# 2. GLM 음이항 회귀 모델 적합 및 summary() 출력
formula = '환자수 ~ 월평균기온 + 월평균습도 + 월평균_일교차 + 월평균_PM25 + sin1 + cos1'
model_full = smf.glm(
    formula, data=l23_df, family=sm.families.NegativeBinomial(alpha=1.0)
).fit()

print('\n' + '=' * 60)
print('✨ [L23] 알레르기성 접촉피부염 GLM 회귀분석 상세 결과 (Summary)')
print('=' * 60)
print(model_full.summary())

L23 유효 데이터 개수: 84개
👉 L23 환자수와 월평균습도의 단순 상관계수: 0.6708

✨ [L23] 알레르기성 접촉피부염 GLM 회귀분석 상세 결과 (Summary)
                 Generalized Linear Model Regression Results                  
Dep. Variable:                    환자수   No. Observations:                   84
Model:                            GLM   Df Residuals:                       77
Model Family:        NegativeBinomial   Df Model:                            6
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -1103.0
Date:                Tue, 04 Aug 2026   Deviance:                      0.29327
Time:                        11:16:18   Pearson chi2:                    0.289
No. Iterations:                     4   Pseudo R-squ. (CS):            0.02162
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------

In [5]:
models_dir = 'models'
os.makedirs(models_dir, exist_ok=True)
scale_target_cols = ['월평균기온', '월평균습도', '월평균_일교차', '월평균_PM25']

results = []
for code, sub_df in df_filtered.groupby('질병코드'):
  name = sub_df['질병명'].iloc[0] if '질병명' in sub_df.columns else code

  sub_df = sub_df.dropna(
      subset=[
          '환자수',
          '월평균기온',
          '월평균습도',
          '월평균_일교차',
          '월평균_PM25',
          'sin1',
          'cos1',
      ]
  )

  if len(sub_df) < 12:
    continue

  formula = (
      '환자수 ~ 월평균기온 + 월평균습도 + 월평균_일교차 + 월평균_PM25 +'
      ' sin1 + cos1'
  )
  formula_null = '환자수 ~ sin1 + cos1'

  try:
    model_full = smf.glm(
        formula, data=sub_df, family=sm.families.NegativeBinomial(alpha=1.0)
    ).fit()
    model_null = smf.glm(
        formula_null,
        data=sub_df,
        family=sm.families.NegativeBinomial(alpha=1.0),
    ).fit()

    dev_full = model_full.deviance
    dev_null = model_null.deviance
    marginal_r2 = max(0, 1 - (dev_full / dev_null)) if dev_null > 0 else 0.0

    params = model_full.params

    w_temp = abs(params.get('월평균기온', 0))
    w_pm25 = abs(params.get('월평균_PM25', 0))
    w_humid = abs(params.get('월평균습도', 0))
    w_dtr = abs(params.get('월평균_일교차', 0))

    tot_w = w_temp + w_pm25 + w_humid + w_dtr
    if tot_w > 0:
      w_temp_pct = round((w_temp / tot_w) * 100, 2)
      w_pm25_pct = round((w_pm25 / tot_w) * 100, 2)
      w_humid_pct = round((w_humid / tot_w) * 100, 2)
      w_dtr_pct = round((w_dtr / tot_w) * 100, 2)
    else:
      w_temp_pct = w_pm25_pct = w_humid_pct = w_dtr_pct = 0.0

    results.append({
        '질병코드': code,
        '질병명': name,
        'Marginal_R2': round(marginal_r2, 4),
        '기온_가중치(%)': w_temp_pct,
        'PM25_가중치(%)': w_pm25_pct,
        '습도_가중치(%)': w_humid_pct,
        '일교차_가중치(%)': w_dtr_pct,
    })

    model_save_data = {
        'params': params,
        'monthly_baseline': sub_df.groupby('월')[scale_target_cols]
        .mean()
        .to_dict(),
        'scaler_mean': sub_df[scale_target_cols].mean().to_dict(),
        'scaler_scale': sub_df[scale_target_cols].std().to_dict(),
    }

    model_file_path = os.path.join(models_dir, f'model_{code}.pkl')
    with open(model_file_path, 'wb') as f:
      pickle.dump(model_save_data, f)

  except Exception as e:
    print(f'[에러] {code}: {e}')

if results:
  result_df = pd.DataFrame(results)
  output_csv = 'weather_weights_summary_2015_onwards.csv'
  result_df.to_csv(output_csv, index=False, encoding='utf-8-sig')
  print(f"\n[성공] '{output_csv}' 저장 완료!")
  display(result_df)  # 주피터 노트북 전용 출력 함수
else:
  print('\n[실패] 결과가 없습니다.')


[성공] 'weather_weights_summary_2015_onwards.csv' 저장 완료!


,질병코드,질병명,Marginal_R2,기온_가중치(%),PM25_가중치(%),습도_가중치(%),일교차_가중치(%)
0,I10,본태성(원발성) 고혈압,0.3840,19.48,21.82,14.35,44.35
1,J06,다발성 및 상세불명 부위의 급성 상기도감염,0.1578,16.99,18.58,10.53,53.90
2,J20,급성 기관지염,0.2116,17.30,22.70,9.86,50.13
3,J30,혈관운동성 및 알레르기성 비염,0.4297,0.43,15.30,10.97,73.30
4,J45,천식,0.1847,56.86,7.49,7.07,28.58
5,K29,위염 및 십이지장염,0.0558,3.50,8.96,21.81,65.73
6,L23,알레르기성 접촉피부염,0.3759,48.92,28.30,1.00,21.78
7,M17,무릎관절증,0.4981,10.12,12.29,13.77,63.82
8,M54,등통증,0.5130,8.56,10.48,13.09,67.86


In [8]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler

# L23 데이터 필터링 및 결측치 제거
l23_df = df_filtered[df_filtered['질병코드'] == 'L23'].copy()
l23_df = l23_df.dropna(
    subset=[
        '환자수',
        '월평균기온',
        '월평균습도',
        '월평균_일교차',
        '월평균_PM25',
        'sin1',
        'cos1',
    ]
)

# 1. 독립변수 스케일링(Standardization) 수행 (가장 중요!)
scale_target_cols = ['월평균기온', '월평균습도', '월평균_일교차', '월평균_PM25']
scaler = StandardScaler()

# 스케일링된 컬럼 생성 (_scaled 붙임)
for col in scale_target_cols:
  l23_df[f'{col}_scaled'] = scaler.fit_transform(l23_df[[col]])

# 2. 스케일링된 변수를 사용하는 새로운 공식(Formula) 정의
formula_scaled = '환자수 ~ 월평균기온_scaled + 월평균습도_scaled + 월평균_일교차_scaled + 월평균_PM25_scaled'

# 3. GLM 음이항 회귀 적합
model_scaled = smf.glm(
    formula_scaled, data=l23_df, family=sm.families.NegativeBinomial(alpha=1.0)
).fit()

print('=' * 60)
print('✨ [L23] 스케일링 적용 후 GLM 회귀분석 상세 결과')
print('=' * 60)
print(model_scaled.summary())

✨ [L23] 스케일링 적용 후 GLM 회귀분석 상세 결과
                 Generalized Linear Model Regression Results                  
Dep. Variable:                    환자수   No. Observations:                   84
Model:                            GLM   Df Residuals:                       79
Model Family:        NegativeBinomial   Df Model:                            4
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -1103.0
Date:                Tue, 04 Aug 2026   Deviance:                      0.30384
Time:                        11:21:18   Pearson chi2:                    0.298
No. Iterations:                     4   Pseudo R-squ. (CS):            0.02150
Covariance Type:            nonrobust                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept

In [9]:
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

# 1. 파일 경로 설정
file_path = 'disease_weather_air_merged.csv'
if not os.path.exists(file_path):
  alt_path = './combined_data/disease_weather_air_merged.csv'
  if os.path.exists(alt_path):
    file_path = alt_path
  else:
    print('파일을 찾을 수 없습니다.')

df = pd.read_csv(file_path)

# 날짜 파싱 및 필터링 (2015년 이후, 코로나 기간 2020~2022 제외)
df['진료년월_dt'] = pd.to_datetime(df['진료년월'].astype(str), errors='coerce')
df['연도'] = df['진료년월_dt'].dt.year
df['월'] = df['진료년월_dt'].dt.month

df_filtered = df[(df['연도'] >= 2015) & (~df['연도'].between(2020, 2022))].copy()

models_dir = 'models_no_season'
os.makedirs(models_dir, exist_ok=True)
scale_target_cols = ['월평균기온', '월평균습도', '월평균_일교차', '월평균_PM25']

results_no_season = []

for code, sub_df in df_filtered.groupby('질병코드'):
  name = sub_df['질병명'].iloc[0] if '질병명' in sub_df.columns else code

  sub_df = sub_df.dropna(subset=['환자수'] + scale_target_cols)

  if len(sub_df) < 12:
    continue

  # 독립변수 표준화 (StandardScaler 적용으로 수치 안정성 및 공정성 확보)
  scaler = StandardScaler()
  scaled_cols = []
  for col in scale_target_cols:
    scaled_col_name = f'{col}_scaled'
    sub_df[scaled_col_name] = scaler.fit_transform(sub_df[[col]])
    scaled_cols.append(scaled_col_name)

  # ✨ 계절성 항(sin1, cos1)을 완전히 배제한 공식 정의
  formula = (
      f"환자수 ~ {scaled_cols[0]} + {scaled_cols[1]} + {scaled_cols[2]} +"
      f' {scaled_cols[3]}'
  )
  formula_null = '환자수 ~ 1'  # 절편만 있는 Null 모델

  try:
    model_full = smf.glm(
        formula, data=sub_df, family=sm.families.NegativeBinomial(alpha=1.0)
    ).fit()
    model_null = smf.glm(
        formula_null,
        data=sub_df,
        family=sm.families.NegativeBinomial(alpha=1.0),
    ).fit()

    dev_full = model_full.deviance
    dev_null = model_null.deviance
    marginal_r2 = max(0, 1 - (dev_full / dev_null)) if dev_null > 0 else 0.0

    params = model_full.params

    # 각 변수별 회귀 계수 절대값 추출
    w_temp = abs(params.get(f'{scale_target_cols[0]}_scaled', 0))
    w_humid = abs(params.get(f'{scale_target_cols[1]}_scaled', 0))
    w_dtr = abs(params.get(f'{scale_target_cols[2]}_scaled', 0))
    w_pm25 = abs(params.get(f'{scale_target_cols[3]}_scaled', 0))

    tot_w = w_temp + w_humid + w_dtr + w_pm25
    if tot_w > 0:
      w_temp_pct = round((w_temp / tot_w) * 100, 2)
      w_humid_pct = round((w_humid / tot_w) * 100, 2)
      w_dtr_pct = round((w_dtr / tot_w) * 100, 2)
      w_pm25_pct = round((w_pm25 / tot_w) * 100, 2)
    else:
      w_temp_pct = w_humid_pct = w_dtr_pct = w_pm25_pct = 0.0

    results_no_season.append({
        '질병코드': code,
        '질병명': name,
        'Marginal_R2': round(marginal_r2, 4),
        '기온_가중치(%)': w_temp_pct,
        'PM25_가중치(%)': w_pm25_pct,
        '습도_가중치(%)': w_humid_pct,
        '일교차_가중치(%)': w_dtr_pct,
    })

    # 모델 저장 데이터 (추후 실시간 예보 점수 산출 시 활용)
    model_save_data = {
        'params': params,
        'monthly_baseline': sub_df.groupby('월')[scale_target_cols]
        .mean()
        .to_dict(),
        'scaler_mean': sub_df[scale_target_cols].mean().to_dict(),
        'scaler_scale': sub_df[scale_target_cols].std().to_dict(),
    }

    model_file_path = os.path.join(models_dir, f'model_{code}.pkl')
    with open(model_file_path, 'wb') as f:
      pickle.dump(model_save_data, f)

  except Exception as e:
    print(f'[에러] {code}: {e}')

if results_no_season:
  result_df_no_season = pd.DataFrame(results_no_season)
  output_csv = 'weather_weights_summary_no_season.csv'
  result_df_no_season.to_csv(output_csv, index=False, encoding='utf-8-sig')
  print(f"\n[성공] '{output_csv}' 저장 완료!")
  display(result_df_no_season)
else:
  print('\n[실패] 결과가 없습니다.')


[성공] 'weather_weights_summary_no_season.csv' 저장 완료!


,질병코드,질병명,Marginal_R2,기온_가중치(%),PM25_가중치(%),습도_가중치(%),일교차_가중치(%)
0,I10,본태성(원발성) 고혈압,0.2875,28.99,32.08,19.06,19.88
1,J06,다발성 및 상세불명 부위의 급성 상기도감염,0.6177,56.46,16.74,13.69,13.12
2,J20,급성 기관지염,0.6040,56.34,21.92,8.59,13.15
3,J30,혈관운동성 및 알레르기성 비염,0.6750,43.39,17.06,18.67,20.88
4,J45,천식,0.5529,40.02,16.26,38.74,4.97
5,K29,위염 및 십이지장염,0.4360,51.02,6.39,29.00,13.59
6,L23,알레르기성 접촉피부염,0.8573,61.10,26.21,8.69,3.99
7,M17,무릎관절증,0.4256,13.72,27.08,23.49,35.71
8,M54,등통증,0.5556,8.29,23.87,31.37,36.47


In [10]:
old_csv_path = 'weather_weights_summary_2015_onwards.csv'
new_csv_path = 'weather_weights_summary_no_season.csv'

if os.path.exists(old_csv_path) and os.path.exists(new_csv_path):
  df_old = pd.read_csv(old_csv_path)
  df_new = pd.read_csv(new_csv_path)

  print('=' * 80)
  print('✨ [비교] L23 (알레르기성 접촉피부염) 가중치 변화 비교')
  print('=' * 80)
  print('1. 기존 모델 (계절성 sin/cos 포함):')
  display(df_old[df_old['질병코드'] == 'L23'])

  print('\n2. 신규 모델 (계절성 항 제거 + 표준화 적용):')
  display(df_new[df_new['질병코드'] == 'L23'])

  print('=' * 80)
  print('📊 전체 질병 요약 비교 (좌측: 기존, 우측: 계절성 제거 후)')
  print('=' * 80)
  merged_df = pd.merge(
      df_old[['질병코드', '질병명', '습도_가중치(%)', '기온_가중치(%)']],
      df_new[['질병코드', '습도_가중치(%)', '기온_가중치(%)']],
      on='질병코드',
      suffixes=('_기존', '_계절성제거'),
  )
  display(merged_df)
else:
  print('비교할 두 CSV 파일이 모두 존재하지 않습니다. 셀 1을 먼저 실행해주세요.')

✨ [비교] L23 (알레르기성 접촉피부염) 가중치 변화 비교
1. 기존 모델 (계절성 sin/cos 포함):


,질병코드,질병명,Marginal_R2,기온_가중치(%),PM25_가중치(%),습도_가중치(%),일교차_가중치(%)
6,L23,알레르기성 접촉피부염,0.3759,48.92,28.3,1.0,21.78



2. 신규 모델 (계절성 항 제거 + 표준화 적용):


,질병코드,질병명,Marginal_R2,기온_가중치(%),PM25_가중치(%),습도_가중치(%),일교차_가중치(%)
6,L23,알레르기성 접촉피부염,0.8573,61.1,26.21,8.69,3.99


📊 전체 질병 요약 비교 (좌측: 기존, 우측: 계절성 제거 후)


,질병코드,질병명,습도_가중치(%)_기존,기온_가중치(%)_기존,습도_가중치(%)_계절성제거,기온_가중치(%)_계절성제거
0,I10,본태성(원발성) 고혈압,14.35,19.48,19.06,28.99
1,J06,다발성 및 상세불명 부위의 급성 상기도감염,10.53,16.99,13.69,56.46
2,J20,급성 기관지염,9.86,17.30,8.59,56.34
3,J30,혈관운동성 및 알레르기성 비염,10.97,0.43,18.67,43.39
4,J45,천식,7.07,56.86,38.74,40.02
5,K29,위염 및 십이지장염,21.81,3.50,29.00,51.02
6,L23,알레르기성 접촉피부염,1.00,48.92,8.69,61.10
7,M17,무릎관절증,13.77,10.12,23.49,13.72
8,M54,등통증,13.09,8.56,31.37,8.29
